# Vessel-Cargo Optimization (Cargill Hackathon)

This notebook implements a CP-SAT optimization model to assign vessels to cargoes, maximizing profit while considering:
- Bunker forward curves for accurate fuel pricing
- Market freight rates for opportunity cost analysis
- Capacity, laycan, and routing constraints

## Port Name Assumption

**All port names are assumed to be already synchronized across all CSVs and `Port Distances.csv`.** The model uses raw port strings without any normalization; any mismatch must be fixed in the source CSVs, not in code.

In [49]:
# Cell 1: Load all CSVs
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import re

base = Path("data")

# Load vessel data
print("Loading vessel data...")
cargill_vessels = pd.read_csv(base / "Cargill_Capesize_Vessels.csv")
market_vessels = pd.read_csv(base / "Market_Vessels_Formatted.csv")

# Load cargo data
print("Loading cargo data...")
cargill_cargoes = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")

# Load port distances
print("Loading port distances...")
port_distances = pd.read_csv(base / "Port Distances.csv")

# Load port locations
print("Loading port locations...")
port_locations = pd.read_csv(base / "port_locations.csv")

# Load bunker forward curve
print("Loading bunker forward curve...")
bunker_forward_curve = pd.read_csv(base / "bunker_forward_curve.csv")

# Load freight rates
print("Loading freight rates...")
freight_rates = pd.read_csv(base / "freight_rates.csv")

print("\n✓ All data loaded successfully!")
print(f"Cargill vessels: {len(cargill_vessels)}")
print(f"Market vessels: {len(market_vessels)}")
print(f"Cargill cargoes: {len(cargill_cargoes)}")
print(f"Market cargoes: {len(market_cargoes)}")
print(f"Port distances: {len(port_distances)}")
print(f"Bunker locations: {len(bunker_forward_curve)}")
print(f"Freight routes: {len(freight_rates)}")

Loading vessel data...
Loading cargo data...
Loading port distances...
Loading port locations...
Loading bunker forward curve...
Loading freight rates...

✓ All data loaded successfully!
Cargill vessels: 4
Market vessels: 11
Cargill cargoes: 3
Market cargoes: 8
Port distances: 15661
Bunker locations: 18
Freight routes: 4


In [50]:
# Diagnostic: Verify raw port names match across all files
print("=" * 80)
print("PORT NAME CONSISTENCY CHECK (Raw Values - No Normalization)")
print("=" * 80)

# Collect unique raw port names from all sources
vessel_ports = set()
for _, row in cargill_vessels_processed.iterrows():
    port = str(row['current_position_port']).strip() if pd.notna(row['current_position_port']) else None
    if port:
        vessel_ports.add(port)
for _, row in market_vessels_processed.iterrows():
    port = str(row['current_position_port']).strip() if pd.notna(row['current_position_port']) else None
    if port:
        vessel_ports.add(port)

cargill_load_ports = set()
cargill_disc_ports = set()
for _, row in cargill_cargoes_processed.iterrows():
    load = str(row['load_port']).strip() if pd.notna(row['load_port']) else None
    disc = str(row['discharge_port']).strip() if pd.notna(row['discharge_port']) else None
    if load:
        cargill_load_ports.add(load)
    if disc:
        cargill_disc_ports.add(disc)

market_load_ports = set()
market_disc_ports = set()
for _, row in market_cargoes_processed.iterrows():
    load = str(row['load_port']).strip() if pd.notna(row['load_port']) else None
    disc = str(row['discharge_port']).strip() if pd.notna(row['discharge_port']) else None
    if load:
        market_load_ports.add(load)
    if disc:
        market_disc_ports.add(disc)

port_locations_ports = set()
for _, row in port_locations.iterrows():
    port = str(row['port_name']).strip() if pd.notna(row['port_name']) else None
    if port:
        port_locations_ports.add(port)

port_distances_ports = set()
for _, row in port_distances.head(100).iterrows():  # Sample first 100
    from_port = str(row['PORT_NAME_FROM']).strip() if pd.notna(row['PORT_NAME_FROM']) else None
    to_port = str(row['PORT_NAME_TO']).strip() if pd.notna(row['PORT_NAME_TO']) else None
    if from_port:
        port_distances_ports.add(from_port)
    if to_port:
        port_distances_ports.add(to_port)

all_ports_in_data = vessel_ports | cargill_load_ports | cargill_disc_ports | market_load_ports | market_disc_ports

print(f"\nUnique ports in vessel current positions: {len(vessel_ports)}")
print(f"  Sample: {sorted(list(vessel_ports))[:10]}")
print(f"\nUnique ports in Cargill cargoes (load): {len(cargill_load_ports)}")
print(f"  Sample: {sorted(list(cargill_load_ports))[:10]}")
print(f"\nUnique ports in Cargill cargoes (discharge): {len(cargill_disc_ports)}")
print(f"  Sample: {sorted(list(cargill_disc_ports))[:10]}")
print(f"\nUnique ports in Market cargoes (load): {len(market_load_ports)}")
print(f"  Sample: {sorted(list(market_load_ports))[:10]}")
print(f"\nUnique ports in Market cargoes (discharge): {len(market_disc_ports)}")
print(f"  Sample: {sorted(list(market_disc_ports))[:10]}")
print(f"\nUnique ports in port_locations.csv: {len(port_locations_ports)}")
print(f"  Sample: {sorted(list(port_locations_ports))[:10]}")
print(f"\nSample ports in Port Distances.csv (first 100 rows): {len(port_distances_ports)}")
print(f"  Sample: {sorted(list(port_distances_ports))[:10]}")

# Check if all ports from data files exist in port_locations
missing_from_port_locations = all_ports_in_data - port_locations_ports
if missing_from_port_locations:
    print(f"\n⚠ WARNING: {len(missing_from_port_locations)} ports in data files not found in port_locations.csv:")
    for p in sorted(missing_from_port_locations)[:20]:
        print(f"  - {p}")
else:
    print(f"\n✓ All ports from data files exist in port_locations.csv")

# Check if ports from data files exist in Port Distances (sample check)
sample_missing = []
for port in list(all_ports_in_data)[:20]:  # Check first 20
    found = False
    for pd_port in port_distances_ports:
        if str(port).strip() == str(pd_port).strip():
            found = True
            break
    if not found:
        sample_missing.append(port)

if sample_missing:
    print(f"\n⚠ WARNING: Sample check found {len(sample_missing)} ports not matching Port Distances format:")
    for p in sample_missing[:10]:
        print(f"  - {p}")
else:
    print(f"\n✓ Sample check: Ports match Port Distances format")

print("\n" + "=" * 80)
print("Note: All port names are used RAW (no normalization).")
print("If mismatches exist, fix them in the source CSVs, not in code.")
print("=" * 80)

PORT NAME CONSISTENCY CHECK (Raw Values - No Normalization)

Unique ports in vessel current positions: 14
  Sample: ['CAOFEIDIAN', 'FANGCHENG', 'GWANGYANG LNG TERMINAL', 'JINGTANG', 'JUBAIL', 'KANDLA', 'MAP TA PHUT', 'MUNDRA', 'PARADIP', 'PORT TALBOT']

Unique ports in Cargill cargoes (load): 3
  Sample: ['ITAGUAI', 'KAMSAR ANCHORAGE', 'PORT HEDLAND']

Unique ports in Cargill cargoes (discharge): 2
  Sample: ['LIANYUNGANG', 'QINGDAO']

Unique ports in Market cargoes (load): 8
  Sample: ['DAMPIER', 'KAMSAR ANCHORAGE', 'PONTA DA MADEIRA', 'PORT HEDLAND', 'SALDANHA BAY', 'TABONEO', 'TUBARAO', 'VANCOUVER (CANADA)']

Unique ports in Market cargoes (discharge): 8
  Sample: ['CAOFEIDIAN', 'FANGCHENG', 'GWANGYANG LNG TERMINAL', 'KRISHNAPATNAM', 'NEW MANGALORE', 'QINGDAO', 'TELUK RUBIAH', 'TIANJIN']

Unique ports in port_locations.csv: 28
  Sample: ['CAOFEIDIAN', 'DAMPIER', 'FANGCHENG', 'GWANGYANG LNG TERMINAL', 'ITAGUAI', 'JINGTANG', 'JUBAIL', 'KAMSAR ANCHORAGE', 'KANDLA', 'KRISHNAPATNAM']

Sa

In [51]:
# Cell 2: Preprocess data (NO PORT NORMALIZATION - using raw CSV values)

# Preprocess vessels
print("Preprocessing vessels...")

# Cargill vessels
cargill_vessels_processed = cargill_vessels.copy()
cargill_vessels_processed['vessel_name'] = cargill_vessels_processed['Vessel Name']
cargill_vessels_processed['deadweight_tonnage_dwt'] = cargill_vessels_processed['DWT (MT)']
cargill_vessels_processed['hire_rate_usd_per_day'] = cargill_vessels_processed['Hire Rate (USD/day)']
cargill_vessels_processed['economical_speed_knots'] = cargill_vessels_processed['Economical Speed Ballast (kn)']  # Use ballast for ballast leg
cargill_vessels_processed['sea_consumption_mt_per_day'] = cargill_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
cargill_vessels_processed['port_consumption_mt_per_day'] = cargill_vessels_processed['Port Consumption Working (mt/day)']
# Use raw port name directly - no normalization
cargill_vessels_processed['current_position_port'] = cargill_vessels_processed['Current Position / Status']
cargill_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(cargill_vessels_processed['ETD'])
cargill_vessels_processed['speed_laden'] = cargill_vessels_processed['Economical Speed Laden (kn)']
cargill_vessels_processed['consumption_laden'] = cargill_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Market vessels (no hire rate column, set to 0 or use market rate)
market_vessels_processed = market_vessels.copy()
market_vessels_processed['vessel_name'] = market_vessels_processed['Vessel Name']
market_vessels_processed['deadweight_tonnage_dwt'] = market_vessels_processed['DWT (MT)']
# Market vessel hire rate: use FFA 5TC rate (cost of chartering a market vessel)
def get_5tc_hire_rate(etd_date):
    tc5_row = freight_rates[freight_rates['route'].str.contains('5TC')]
    if len(tc5_row) == 0:
        return 18454
    tc5_row = tc5_row.iloc[0]
    if isinstance(etd_date, str):
        etd_date = pd.to_datetime(etd_date)
    month_key = f"{etd_date.year}-{etd_date.month:02d}"
    if month_key in tc5_row.index and pd.notna(tc5_row[month_key]):
        return float(tc5_row[month_key])
    quarter = (etd_date.month - 1) // 3 + 1
    q_key = f"{etd_date.year}-Q{quarter}"
    if q_key in tc5_row.index and pd.notna(tc5_row[q_key]):
        return float(tc5_row[q_key])
    return 18454

market_vessels_processed['hire_rate_usd_per_day'] = market_vessels_processed['ETD'].apply(
    lambda etd: get_5tc_hire_rate(pd.to_datetime(etd))
)
print("Market vessel hire rates (FFA 5TC):")
for _, v in market_vessels_processed.iterrows():
    print(f"  {v['vessel_name']}: ${v['hire_rate_usd_per_day']:,.0f}/day")
market_vessels_processed['economical_speed_knots'] = market_vessels_processed['Economical Speed Ballast (kn)']
market_vessels_processed['sea_consumption_mt_per_day'] = market_vessels_processed['Economical Sea Consumption Ballast (mt/day)']
market_vessels_processed['port_consumption_mt_per_day'] = market_vessels_processed['Port Consumption Working (mt/day)']
# Use raw port name directly - no normalization
market_vessels_processed['current_position_port'] = market_vessels_processed['Current Position / Status']
market_vessels_processed['estimated_time_of_departure'] = pd.to_datetime(market_vessels_processed['ETD'])
market_vessels_processed['speed_laden'] = market_vessels_processed['Economical Speed Laden (kn)']
market_vessels_processed['consumption_laden'] = market_vessels_processed['Economical Sea Consumption Laden (mt/day)']

# Preprocess cargoes
print("Preprocessing cargoes...")

# Cargill cargoes
cargill_cargoes_processed = cargill_cargoes.copy()
cargill_cargoes_processed['cargo_id'] = ['CARGILL_' + str(i+1) for i in range(len(cargill_cargoes_processed))]
cargill_cargoes_processed['quantity_mt'] = cargill_cargoes_processed['Quantity_MT']
cargill_cargoes_processed['freight_rate_usd_per_mt'] = cargill_cargoes_processed['Freight_Rate_USD_PMT']
cargill_cargoes_processed['commission_percent'] = cargill_cargoes_processed['Commission_Percent'] / 100.0
cargill_cargoes_processed['laycan_start_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_Start'])
cargill_cargoes_processed['laycan_end_date'] = pd.to_datetime(cargill_cargoes_processed['Laycan_End'])
# Use raw port names directly - no normalization
cargill_cargoes_processed['load_port'] = cargill_cargoes_processed['Load_Port_Primary']
cargill_cargoes_processed['discharge_port'] = cargill_cargoes_processed['Discharge_Port_Primary']
cargill_cargoes_processed['port_costs_usd'] = cargill_cargoes_processed['Port_Cost_USD'].fillna(0)
cargill_cargoes_processed['load_turn_time_hours'] = cargill_cargoes_processed['Load_Turn_Time_Hours'].fillna(12)
cargill_cargoes_processed['discharge_turn_time_hours'] = cargill_cargoes_processed['Discharge_Turn_Time_Hours'].fillna(24)
cargill_cargoes_processed['route'] = cargill_cargoes_processed['Route']

# Market cargoes
market_cargoes_processed = market_cargoes.copy()
market_cargoes_processed['cargo_id'] = ['MARKET_' + str(i+1) for i in range(len(market_cargoes_processed))]
market_cargoes_processed['quantity_mt'] = market_cargoes_processed['Quantity_MT']
# Market cargoes don't have freight rate - will use market rates
market_cargoes_processed['freight_rate_usd_per_mt'] = 0  # Will be filled from freight_rates.csv
market_cargoes_processed['commission_percent'] = market_cargoes_processed['Commission_Percent'] / 100.0
market_cargoes_processed['laycan_start_date'] = pd.to_datetime(market_cargoes_processed['Laycan_Start'])
market_cargoes_processed['laycan_end_date'] = pd.to_datetime(market_cargoes_processed['Laycan_End'])
# Use raw port names directly - no normalization
market_cargoes_processed['load_port'] = market_cargoes_processed['Load_Port']
market_cargoes_processed['discharge_port'] = market_cargoes_processed['Discharge_Port']
# Sum load and discharge port costs
market_cargoes_processed['port_costs_usd'] = (
    market_cargoes_processed['Port_Cost_Load_USD'].fillna(0) + 
    market_cargoes_processed['Port_Cost_Discharge_USD'].fillna(0)
)
market_cargoes_processed['load_turn_time_hours'] = market_cargoes_processed['Load_Turn_Time_hr'].fillna(12)
market_cargoes_processed['discharge_turn_time_hours'] = market_cargoes_processed['Discharge_Turn_Time_hr'].fillna(24)
market_cargoes_processed['route'] = market_cargoes_processed['Route']

# Build distance lookup from Port Distances.csv using raw port names
print("Building distance lookup from Port Distances.csv...")
port_distances_processed = port_distances.copy()
# Use raw port names - only strip whitespace for safety
port_distances_processed['port_from'] = port_distances_processed['PORT_NAME_FROM'].str.strip()
port_distances_processed['port_to'] = port_distances_processed['PORT_NAME_TO'].str.strip()
port_distances_processed['distance_nautical_miles'] = port_distances_processed['DISTANCE']

# Create distance lookup dictionary for fast access
# Using raw port names exactly as they appear in Port Distances.csv
distance_lookup = {}
for _, row in port_distances_processed.iterrows():
    port_from = row['port_from']
    port_to = row['port_to']
    distance = row['distance_nautical_miles']
    
    # Store with raw port names
    key = (port_from, port_to)
    distance_lookup[key] = distance
    
    # Also store reverse direction
    key_reverse = (port_to, port_from)
    if key_reverse not in distance_lookup:
        distance_lookup[key_reverse] = distance

print("\n✓ Data preprocessing complete!")
print(f"Processed {len(cargill_vessels_processed)} Cargill vessels")
print(f"Processed {len(market_vessels_processed)} market vessels")
print(f"Processed {len(cargill_cargoes_processed)} Cargill cargoes")
print(f"Processed {len(market_cargoes_processed)} market cargoes")
print(f"Distance lookup entries: {len(distance_lookup)}")


Preprocessing vessels...
Preprocessing cargoes...
Building distance lookup from Port Distances.csv...

✓ Data preprocessing complete!
Processed 4 Cargill vessels
Processed 11 market vessels
Processed 3 Cargill cargoes
Processed 8 market cargoes
Distance lookup entries: 30896


In [52]:
# Cell 3: Build bunker price lookup function

# Reshape bunker forward curve from wide to long format
bunker_long = []
date_cols = [col for col in bunker_forward_curve.columns if re.match(r'^\d{4}-\d{2}-\d{2}$', col)]

for _, row in bunker_forward_curve.iterrows():
    location = row['location']
    fuel_grade = row['fuel_grade']
    lat = row['latitude']
    lon = row['longitude']
    for date_col in date_cols:
        price = row[date_col]
        if pd.notna(price):
            date = pd.to_datetime(date_col)
            bunker_long.append({
                'location': location,
                'fuel_grade': fuel_grade,
                'latitude': lat,
                'longitude': lon,
                'date': date,
                'price': price
            })

bunker_df = pd.DataFrame(bunker_long)
bunker_df = bunker_df.sort_values(['location', 'fuel_grade', 'date'])

print(f"Bunker data points: {len(bunker_df)}")
print(f"Locations: {bunker_df['location'].nunique()}")
print(f"Date range: {bunker_df['date'].min()} to {bunker_df['date'].max()}")

# Create location mapping from port names to bunker locations
# Use port_locations to match ports to bunker curve locations by lat/lon proximity
# Using raw port names from port_locations.csv (no normalization)
port_to_bunker_location = {}
for _, port_row in port_locations.iterrows():
    port_name = str(port_row['port_name']).strip()  # Use raw port name, only strip whitespace
    port_lat = port_row['latitude']
    port_lon = port_row['longitude']
    
    # Find nearest bunker location by distance
    min_dist = float('inf')
    nearest_location = None
    for _, bunker_row in bunker_forward_curve.iterrows():
        bunker_lat = bunker_row['latitude']
        bunker_lon = bunker_row['longitude']
        # Simple Euclidean distance (approximate)
        dist = ((port_lat - bunker_lat)**2 + (port_lon - bunker_lon)**2)**0.5
        if dist < min_dist:
            min_dist = dist
            nearest_location = bunker_row['location']
    
    # Map raw port name to bunker location
    port_to_bunker_location[port_name] = nearest_location

def get_bunker_price(port_name, fuel_grade='VLSFO', date=None):
    """
    Get bunker price for a port and date.
    
    Args:
        port_name: Port name (raw string from CSV, no normalization)
        fuel_grade: 'VLSFO' or 'MGO'
        date: datetime object or date
    
    Returns:
        Price in USD/MT, or None if not found
    """
    if date is None:
        return None
    
    if isinstance(date, datetime):
        date = date.date()
    elif isinstance(date, pd.Timestamp):
        date = date.date()
    
    # Use raw port name - only strip whitespace for safety
    port_clean = str(port_name).strip() if port_name else None
    if not port_clean:
        return None
    
    # Try to find bunker location using raw port name
    bunker_location = None
    if port_clean in port_to_bunker_location:
        bunker_location = port_to_bunker_location[port_clean]
    
    if not bunker_location:
        # Fallback: use average price across all locations
        matching = bunker_df[(bunker_df['fuel_grade'] == fuel_grade) & (bunker_df['date'] == pd.Timestamp(date))]
        if len(matching) > 0:
            return matching['price'].mean()
        return None
    
    # Get prices for this location and fuel grade
    location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                 (bunker_df['fuel_grade'] == fuel_grade)].copy()
    
    if len(location_prices) == 0:
        # Try VLSFO if MGO not available
        if fuel_grade == 'MGO':
            location_prices = bunker_df[(bunker_df['location'] == bunker_location) & 
                                         (bunker_df['fuel_grade'] == 'VLSFO')].copy()
        if len(location_prices) == 0:
            return None
    
    # Interpolate between dates
    location_prices = location_prices.sort_values('date')
    target_date = pd.Timestamp(date)
    
    # Exact match
    exact = location_prices[location_prices['date'] == target_date]
    if len(exact) > 0:
        return exact.iloc[0]['price']
    
    # Before first date
    if target_date < location_prices['date'].min():
        return location_prices.iloc[0]['price']
    
    # After last date
    if target_date > location_prices['date'].max():
        return location_prices.iloc[-1]['price']
    
    # Interpolate between two dates
    before = location_prices[location_prices['date'] < target_date].iloc[-1]
    after = location_prices[location_prices['date'] > target_date].iloc[0]
    
    days_between = (after['date'] - before['date']).days
    days_to_target = (target_date - before['date']).days
    
    if days_between == 0:
        return before['price']
    
    weight = days_to_target / days_between
    interpolated_price = before['price'] * (1 - weight) + after['price'] * weight
    
    return interpolated_price

# Test the function
test_date = pd.to_datetime('2026-03-15')
# Test with raw port name
test_price = get_bunker_price('QINGDAO', 'VLSFO', test_date)
print(f"\nTest: Bunker price for QINGDAO on {test_date.date()}: ${test_price:.2f}/MT")
print("✓ Bunker price lookup function ready")

Bunker data points: 216
Locations: 9
Date range: 2026-02-01 00:00:00 to 2027-01-01 00:00:00

Test: Bunker price for QINGDAO on 2026-03-15: $641.19/MT
✓ Bunker price lookup function ready


In [53]:
# Cell 4: Build freight rate matching and lookup function

# Reshape freight rates from wide to long format
freight_long = []
period_cols = [col for col in freight_rates.columns if col != 'route']

for _, row in freight_rates.iterrows():
    route = row['route']
    for period_col in period_cols:
        rate = row[period_col]
        if pd.notna(rate) and rate != '':
            freight_long.append({
                'route': route,
                'period': period_col,
                'rate': rate
            })

freight_rates_long = pd.DataFrame(freight_long)

print(f"Freight rate data points: {len(freight_rates_long)}")
print(f"Routes: {freight_rates_long['route'].nunique()}")

# Route matching logic
def match_route_to_freight_rate(cargo_route):
    """
    Match a cargo route (e.g., "Brazil – China") to a freight rate route.
    
    Returns:
        Matched route name from freight_rates.csv, or None
    """
    if pd.isna(cargo_route):
        return None
    
    route_str = str(cargo_route).strip()
    
    # Manual mapping for common routes
    route_mapping = {
        'Brazil – China': 'C3 (Tubarao-Qingdao)',
        'Brazil – China (Iron Ore)': 'C3 (Tubarao-Qingdao)',
        'Australia – China': 'C5 (West Australia-Qingdao)',
        'Australia – China (Iron Ore)': 'C5 (West Australia-Qingdao)',
        'West Africa – China': None,  # No direct match
        'West Africa – India': None,
        'South Africa – China': None,
        'Indonesia – India': None,
        'Canada – China': None,
        'Australia – South Korea': None,
        'Brazil – Malaysia': None,
    }
    
    if route_str in route_mapping:
        return route_mapping[route_str]
    
    # Try pattern matching
    route_upper = route_str.upper()
    for freight_route in freight_rates['route'].unique():
        freight_route_clean = str(freight_route).strip().upper()
        # Check if key terms match
        if 'BRAZIL' in route_upper and 'CHINA' in route_upper and 'C3' in freight_route_clean:
            return freight_route
        if 'AUSTRALIA' in route_upper and 'CHINA' in route_upper and 'C5' in freight_route_clean:
            return freight_route
    
    return None

def get_market_freight_rate(cargo_route, laycan_date):
    """
    Get market freight rate for a cargo route and laycan period.
    
    Args:
        cargo_route: Route string from cargo data
        laycan_date: datetime or date for the laycan
    
    Returns:
        Market freight rate in USD/MT, or None
    """
    matched_route = match_route_to_freight_rate(cargo_route)
    if not matched_route:
        return None
    
    if isinstance(laycan_date, datetime):
        laycan_date = laycan_date.date()
    elif isinstance(laycan_date, pd.Timestamp):
        laycan_date = laycan_date.date()
    
    # Match period
    year = laycan_date.year
    month = laycan_date.month
    quarter = (month - 1) // 3 + 1
    
    # Try to match period columns
    period_candidates = [
        f"{year}-{month:02d}",  # e.g., "2026-03"
        f"{year}-Q{quarter}",   # e.g., "2026-Q1"
        str(year),               # e.g., "2026"
    ]
    
    for period in period_candidates:
        matching = freight_rates_long[
            (freight_rates_long['route'] == matched_route) & 
            (freight_rates_long['period'] == period)
        ]
        if len(matching) > 0:
            return matching.iloc[0]['rate']
    
    # If no exact match, try to find closest period
    route_rates = freight_rates_long[freight_rates_long['route'] == matched_route]
    if len(route_rates) > 0:
        # Return average or first available
        return route_rates['rate'].iloc[0]
    
    return None

# Test the function
test_route = "Brazil – China"
test_laycan = pd.to_datetime('2026-03-15')
test_rate = get_market_freight_rate(test_route, test_laycan)
print(f"\nTest: Market freight rate for '{test_route}' in {test_laycan.date()}: ${test_rate:.2f}/MT")
print("✓ Freight rate lookup function ready")

Freight rate data points: 49
Routes: 4

Test: Market freight rate for 'Brazil – China' in 2026-03-15: $20908.00/MT
✓ Freight rate lookup function ready


In [54]:
# Cell 5: Implement compute_profit function

def compute_profit(vessel_row, cargo_row, distance_lookup, get_bunker_price_fn, get_market_freight_rate_fn):
    """
    Compute profit for a vessel-cargo assignment.
    
    Returns:
        dict with keys: 'profit', 'revenue', 'costs', 'opportunity_cost', 'market_rate', 
        'days_ballast', 'days_laden', 'total_days', or None if infeasible
    """
    try:
        # Feasibility checks
        # 1. Capacity check
        if cargo_row['quantity_mt'] > vessel_row['deadweight_tonnage_dwt']:
            return None
        
        # 2. Distance lookups
        ballast_port_from = vessel_row['current_position_port']
        ballast_port_to = cargo_row['load_port']
        laden_port_from = cargo_row['load_port']
        laden_port_to = cargo_row['discharge_port']
        
        if not ballast_port_from or not ballast_port_to or not laden_port_to:
            return None
        
        # Helper function to get distance - using raw port names
        def get_distance(port_from, port_to):
            """
            Get distance between two ports using raw port names from CSVs.
            Only strips whitespace for safety - no other normalization.
            """
            if not port_from or not port_to:
                return None
            
            # Use raw port names - only strip whitespace
            port_from_clean = str(port_from).strip()
            port_to_clean = str(port_to).strip()
            
            # Try exact match (forward direction)
            distance = distance_lookup.get((port_from_clean, port_to_clean))
            if distance is not None:
                return distance
            
            # Try reverse direction
            distance = distance_lookup.get((port_to_clean, port_from_clean))
            if distance is not None:
                return distance
            
            return None
        
        ballast_distance = get_distance(ballast_port_from, ballast_port_to)
        laden_distance = get_distance(laden_port_from, laden_port_to)
        
        if ballast_distance is None or laden_distance is None:
            return None
        
        # 3. Time calculations
        # Ballast leg
        ballast_speed = vessel_row['economical_speed_knots']
        days_ballast = ballast_distance / (ballast_speed * 24.0)
        
        # Laden leg
        laden_speed = vessel_row['speed_laden']
        days_laden = laden_distance / (laden_speed * 24.0)
        
        # Port time
        port_days = (cargo_row['load_turn_time_hours'] + cargo_row['discharge_turn_time_hours']) / 24.0
        
        total_days = days_ballast + days_laden + port_days
        
        # 4. Laycan check
        etd = vessel_row['estimated_time_of_departure']
        if isinstance(etd, str):
            etd = pd.to_datetime(etd)
        arrival_at_load = etd + timedelta(days=days_ballast)
        
        if arrival_at_load.date() > cargo_row['laycan_end_date'].date():
            return None
        
        # 5. Revenue
        freight_rate = cargo_row['freight_rate_usd_per_mt']
        quantity = cargo_row['quantity_mt']
        revenue = freight_rate * quantity
        
        # 6. Costs
        # Commission
        commission = cargo_row['commission_percent'] * revenue
        
        # Fuel costs
        # Ballast fuel
        fuel_ballast = vessel_row['sea_consumption_mt_per_day'] * days_ballast
        bunker_price_ballast = get_bunker_price_fn(ballast_port_from, 'VLSFO', etd.date())
        if bunker_price_ballast is None:
            bunker_price_ballast = 500  # Default fallback
        
        # Laden fuel
        fuel_laden = vessel_row['consumption_laden'] * days_laden
        bunker_price_laden = get_bunker_price_fn(laden_port_from, 'VLSFO', arrival_at_load.date())
        if bunker_price_laden is None:
            bunker_price_laden = 500  # Default fallback
        
        # Port fuel
        fuel_port = vessel_row['port_consumption_mt_per_day'] * port_days
        bunker_price_port = get_bunker_price_fn(laden_port_from, 'VLSFO', arrival_at_load.date())
        if bunker_price_port is None:
            bunker_price_port = 500  # Default fallback
        
        fuel_cost = (fuel_ballast * bunker_price_ballast + 
                    fuel_laden * bunker_price_laden + 
                    fuel_port * bunker_price_port)
        
        # Hire cost
        hire_cost = vessel_row['hire_rate_usd_per_day'] * total_days
        
        # Port costs
        port_costs = cargo_row['port_costs_usd']
        
        total_costs = commission + fuel_cost + hire_cost + port_costs
        
        # Profit
        profit = revenue - total_costs
        
        # 7. Opportunity cost (market rate comparison)
        market_rate = get_market_freight_rate_fn(cargo_row.get('route'), cargo_row['laycan_start_date'])
        opportunity_cost = None
        if market_rate is not None:
            market_revenue = market_rate * quantity
            opportunity_cost = market_revenue - revenue  # Positive if market rate is higher
        
        return {
            'profit': profit,
            'revenue': revenue,
            'costs': total_costs,
            'commission': commission,
            'fuel_cost': fuel_cost,
            'hire_cost': hire_cost,
            'port_costs': port_costs,
            'opportunity_cost': opportunity_cost,
            'market_rate': market_rate,
            'offered_rate': freight_rate,
            'days_ballast': days_ballast,
            'days_laden': days_laden,
            'total_days': total_days,
            'ballast_distance': ballast_distance,
            'laden_distance': laden_distance
        }
    
    except Exception as e:
        print(f"Error computing profit: {e}")
        return None

print("✓ compute_profit function implemented")

✓ compute_profit function implemented


In [55]:
# Install ortools
%pip install ortools

Python(11936) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Note: you may need to restart the kernel to use updated packages.


In [56]:
# Cell 6: Implement optimize_assignments using OR-Tools CP-SAT

from ortools.sat.python import cp_model

def optimize_assignments(cargill_vessels_df, market_vessels_df, 
                         cargill_cargoes_df, market_cargoes_df,
                         distance_lookup, get_bunker_price_fn, get_market_freight_rate_fn):
    """
    Optimize vessel-cargo assignments using CP-SAT.
    
    Returns:
        assignments_df: DataFrame with optimal assignments
        total_profit: Total profit from all assignments
    """
    model = cp_model.CpModel()
    
    # Build all feasible pairs and compute profits
    feasible_pairs = []
    
    # Cargill vessels -> Cargill cargoes
    for v_idx, vessel in cargill_vessels_df.iterrows():
        for c_idx, cargo in cargill_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup, 
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'cargill',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'committed',
                    'profit_data': profit_data
                })
    
    # Cargill vessels -> Market cargoes
    for v_idx, vessel in cargill_vessels_df.iterrows():
        for c_idx, cargo in market_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup,
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'cargill',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'market',
                    'profit_data': profit_data
                })
    
    # Market vessels -> Cargill cargoes
    for v_idx, vessel in market_vessels_df.iterrows():
        for c_idx, cargo in cargill_cargoes_df.iterrows():
            profit_data = compute_profit(vessel, cargo, distance_lookup,
                                         get_bunker_price_fn, get_market_freight_rate_fn)
            if profit_data is not None:
                feasible_pairs.append({
                    'origin': 'market',
                    'vessel_idx': v_idx,
                    'cargo_idx': c_idx,
                    'cargo_type': 'committed',
                    'profit_data': profit_data
                })
    
    print(f"Found {len(feasible_pairs)} feasible vessel-cargo pairs")
    
    if len(feasible_pairs) == 0:
        print("No feasible pairs found!")
        return pd.DataFrame(), 0
    
    # Create decision variables
    variables = {}
    for i, pair in enumerate(feasible_pairs):
        var_name = f"x_{pair['origin']}_{pair['vessel_idx']}_{pair['cargo_type']}_{pair['cargo_idx']}"
        variables[i] = model.NewBoolVar(var_name)
    
    # Constraints
    # 1. Each committed cargo must be assigned exactly once
    # Ensure ALL committed cargoes are covered (from any vessel origin)
    for cargo_idx in range(len(cargill_cargoes_df)):
        # Find all pairs for this committed cargo (from Cargill or Market vessels)
        pairs_for_cargo = [i for i, p in enumerate(feasible_pairs) 
                          if p['cargo_type'] == 'committed' and p['cargo_idx'] == cargo_idx]
        if len(pairs_for_cargo) > 0:
            model.Add(sum(variables[i] for i in pairs_for_cargo) == 1)
        else:
            # If no feasible pairs exist, the model will be infeasible
            print(f"WARNING: Committed cargo {cargo_idx} (CARGILL_{cargo_idx+1}) has no feasible vessel assignments!")
    
    # 2. Each market cargo can be assigned at most once
    # Each market cargo can be assigned at most once (across ALL vessel types)
    market_cargo_indices = set()
    for i, pair in enumerate(feasible_pairs):
        if pair['cargo_type'] == 'market':
            market_cargo_indices.add(pair['cargo_idx'])
    
    for cargo_idx in market_cargo_indices:
        pairs_for_cargo = [i for i, p in enumerate(feasible_pairs) 
                          if p['cargo_type'] == 'market' and p['cargo_idx'] == cargo_idx]
        if len(pairs_for_cargo) > 0:
            model.Add(sum(variables[i] for i in pairs_for_cargo) <= 1)
    
    # 3. Each vessel can be assigned at most once
    vessel_indices = set()
    for i, pair in enumerate(feasible_pairs):
        vessel_indices.add((pair['origin'], pair['vessel_idx']))
    
    for (origin, vessel_idx) in vessel_indices:
        pairs_for_vessel = [i for i, p in enumerate(feasible_pairs) 
                           if p['origin'] == origin and p['vessel_idx'] == vessel_idx]
        if len(pairs_for_vessel) > 0:
            model.Add(sum(variables[i] for i in pairs_for_vessel) <= 1)
    
    # Objective: Maximize total profit
    # Scale profit to integers (multiply by 100)
    objective_terms = []
    for i, pair in enumerate(feasible_pairs):
        profit_scaled = int(round(pair['profit_data']['profit'] * 100))
        objective_terms.append(variables[i] * profit_scaled)
    
    model.Maximize(sum(objective_terms))
    
    # Solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 300.0  # 5 minutes
    status = solver.Solve(model)
    
    if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
        print(f"Solution status: {'OPTIMAL' if status == cp_model.OPTIMAL else 'FEASIBLE'}")
        
        # Extract assignments
        assignments = []
        total_profit = 0
        
        for i, pair in enumerate(feasible_pairs):
            if solver.Value(variables[i]) == 1:
                profit_data = pair['profit_data']
                total_profit += profit_data['profit']
                
                # Get vessel and cargo details
                if pair['origin'] == 'cargill':
                    vessel = cargill_vessels_df.loc[pair['vessel_idx']]
                    vessel_name = vessel['vessel_name']
                else:
                    vessel = market_vessels_df.loc[pair['vessel_idx']]
                    vessel_name = vessel['vessel_name']
                
                if pair['cargo_type'] == 'committed':
                    cargo = cargill_cargoes_df.loc[pair['cargo_idx']]
                    cargo_id = cargo['cargo_id']
                    load_port = cargo['load_port']
                    discharge_port = cargo['discharge_port']
                else:
                    cargo = market_cargoes_df.loc[pair['cargo_idx']]
                    cargo_id = cargo['cargo_id']
                    load_port = cargo['load_port']
                    discharge_port = cargo['discharge_port']
                
                assignments.append({
                    'Vessel_Type': 'Cargill' if pair['origin'] == 'cargill' else 'Market',
                    'Vessel_Name': vessel_name,
                    'Cargo_Type': 'Committed' if pair['cargo_type'] == 'committed' else 'Market',
                    'Cargo_ID': cargo_id,
                    'Load_Port': load_port,
                    'Discharge_Port': discharge_port,
                    'Profit': profit_data['profit'],
                    'Revenue': profit_data['revenue'],
                    'Costs': profit_data['costs'],
                    'TCE': profit_data['profit'] / profit_data['total_days'] if profit_data['total_days'] > 0 else 0,
                    'Voyage_Days': profit_data['total_days'],
                    'Days_Ballast': profit_data['days_ballast'],
                    'Days_Laden': profit_data['days_laden'],
                    'Offered_Rate': profit_data['offered_rate'],
                    'Market_Rate': profit_data['market_rate'],
                    'Opportunity_Cost': profit_data['opportunity_cost'],
                    'Ballast_Distance': profit_data['ballast_distance'],
                    'Laden_Distance': profit_data['laden_distance']
                })
        
        assignments_df = pd.DataFrame(assignments)
        return assignments_df, total_profit
    
    else:
        print(f"Solution status: {solver.StatusName(status)}")
        return pd.DataFrame(), 0

print("✓ optimize_assignments function implemented")


✓ optimize_assignments function implemented


In [57]:
# Comprehensive Diagnostic: Port Normalization and Distance Lookup
print("=" * 80)
print("COMPREHENSIVE PORT NORMALIZATION & DISTANCE DIAGNOSTIC")
print("=" * 80)

# Test the actual get_distance function from compute_profit
import unicodedata

def normalize_for_distance(port_name):
    """Same function as in compute_profit"""
    if not port_name:
        return None
    nfd = unicodedata.normalize('NFD', port_name)
    no_accents = ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')
    normalized = no_accents.upper()
    if '_' in normalized:
        normalized = normalized.split('_')[0]
    return normalized.strip()

def test_distance_lookup(port_from, port_to, show_details=True):
    """Test distance lookup with same logic as get_distance in compute_profit"""
    if not port_from or not port_to:
        return None, "missing port"
    
    # Try exact match
    distance = distance_lookup.get((port_from, port_to))
    if distance is not None:
        return distance, "exact match"
    
    # Extract port name only
    port_from_base = port_from.split('_')[0] if '_' in port_from else port_from
    port_to_base = port_to.split('_')[0] if '_' in port_to else port_to
    
    # Try port-only match
    distance = distance_lookup.get((port_from_base, port_to_base))
    if distance is not None:
        return distance, f"port-only ({port_from_base} -> {port_to_base})"
    
    # Try with accents removed
    port_from_norm = normalize_for_distance(port_from_base)
    port_to_norm = normalize_for_distance(port_to_base)
    
    if port_from_norm and port_to_norm:
        distance = distance_lookup.get((port_from_norm, port_to_norm))
        if distance is not None:
            return distance, f"accent-normalized ({port_from_norm} -> {port_to_norm})"
        
        # Try reverse
        distance = distance_lookup.get((port_to_norm, port_from_norm))
        if distance is not None:
            return distance, f"accent-normalized reverse ({port_to_norm} -> {port_from_norm})"
    
    # Try reverse (original)
    distance = distance_lookup.get((port_to_base, port_from_base))
    if distance is not None:
        return distance, f"reverse ({port_to_base} -> {port_from_base})"
    
    return None, "not found"

# Check CARGILL_3 port normalization
print("\n1. PORT NORMALIZATION CHECK")
print("-" * 80)
cargo_idx = 2
cargo = cargill_cargoes_processed.iloc[cargo_idx]
print(f"Cargo: {cargo['cargo_id']}")
print(f"Original Load Port: {cargill_cargoes.iloc[cargo_idx]['Load_Port_Primary']}")
print(f"Normalized Load Port: {cargo['load_port']}")
print(f"  → Port name only: {cargo['load_port'].split('_')[0] if '_' in cargo['load_port'] else cargo['load_port']}")
print(f"  → Accent-normalized: {normalize_for_distance(cargo['load_port'])}")

print(f"\nOriginal Discharge Port: {cargill_cargoes.iloc[cargo_idx]['Discharge_Port_Primary']}")
print(f"Normalized Discharge Port: {cargo['discharge_port']}")
print(f"  → Port name only: {cargo['discharge_port'].split('_')[0] if '_' in cargo['discharge_port'] else cargo['discharge_port']}")
print(f"  → Accent-normalized: {normalize_for_distance(cargo['discharge_port'])}")

# Check distance lookups for CARGILL_3
print("\n2. DISTANCE LOOKUP TEST FOR CARGILL_3")
print("-" * 80)
load_port = cargo['load_port']
discharge_port = cargo['discharge_port']

# Test laden leg (ITAGUAÍ_BRAZIL -> QINGDAO_CHINA)
print(f"\nLaden Leg: {load_port} -> {discharge_port}")
laden_dist, laden_method = test_distance_lookup(load_port, discharge_port)
if laden_dist:
    print(f"  ✓ Found: {laden_dist:.1f} nm ({laden_method})")
else:
    print(f"  ❌ Not found ({laden_method})")
    # Show what's in distance_lookup
    print(f"  Checking variations in distance_lookup:")
    print(f"    ('{load_port}', '{discharge_port}'): {distance_lookup.get((load_port, discharge_port))}")
    load_base = load_port.split('_')[0] if '_' in load_port else load_port
    disc_base = discharge_port.split('_')[0] if '_' in discharge_port else discharge_port
    print(f"    ('{load_base}', '{disc_base}'): {distance_lookup.get((load_base, disc_base))}")
    load_norm = normalize_for_distance(load_base)
    disc_norm = normalize_for_distance(disc_base)
    print(f"    ('{load_norm}', '{disc_norm}'): {distance_lookup.get((load_norm, disc_norm))}")

# Check all vessels (Cargill + Market)
print("\n3. VESSEL ASSIGNMENT CHECK (Cargill + Market)")
print("-" * 80)

all_vessels = []
for v_idx, vessel in cargill_vessels_processed.iterrows():
    all_vessels.append(('Cargill', v_idx, vessel))
for v_idx, vessel in market_vessels_processed.iterrows():
    all_vessels.append(('Market', v_idx, vessel))

feasible_count = 0
for vessel_type, v_idx, vessel in all_vessels:
    print(f"\n  {vessel_type} Vessel: {vessel['vessel_name']}")
    print(f"    Current position: {vessel['current_position_port']}")
    print(f"    ETD: {vessel['estimated_time_of_departure'].date()}")
    
    # Check ballast distance
    ballast_from = vessel['current_position_port']
    ballast_to = load_port
    ballast_dist, ballast_method = test_distance_lookup(ballast_from, ballast_to)
    
    # Check laden distance
    laden_dist, laden_method = test_distance_lookup(load_port, discharge_port)
    
    if ballast_dist and laden_dist:
        print(f"    ✓ Ballast: {ballast_dist:.1f} nm ({ballast_method})")
        print(f"    ✓ Laden: {laden_dist:.1f} nm ({laden_method})")
        
        # Calculate arrival
        days_ballast = ballast_dist / (vessel['economical_speed_knots'] * 24.0)
        arrival = vessel['estimated_time_of_departure'] + timedelta(days=days_ballast)
        print(f"    Arrival: {arrival.date()}")
        
        # Check profit
        profit_data = compute_profit(vessel, cargo, distance_lookup, get_bunker_price, get_market_freight_rate)
        if profit_data:
            print(f"    ✓ PROFIT: ${profit_data['profit']:,.2f}")
            feasible_count += 1
        else:
            print(f"    ❌ PROFIT: None (infeasible)")
    else:
        print(f"    ❌ Ballast: {ballast_method if not ballast_dist else 'OK'}")
        print(f"    ❌ Laden: {laden_method if not laden_dist else 'OK'}")

print(f"\n  Total feasible vessels: {feasible_count}")

# Check what optimization actually chose
print("\n4. OPTIMIZATION RESULTS CHECK")
print("-" * 80)
if 'assignments_df' in globals() and len(assignments_df) > 0:
    cargill3_assignments = assignments_df[assignments_df['Cargo_ID'] == 'CARGILL_3']
    if len(cargill3_assignments) > 0:
        print(f"✓ CARGILL_3 was assigned:")
        for _, assignment in cargill3_assignments.iterrows():
            print(f"  Vessel: {assignment['Vessel_Type']} - {assignment['Vessel_Name']}")
            print(f"  Profit: ${assignment['Profit']:,.2f}")
            print(f"  Load: {assignment['Load_Port']} -> Discharge: {assignment['Discharge_Port']}")
    else:
        print("❌ CARGILL_3 was NOT assigned in optimization")
else:
    print("⚠ Optimization not run yet - run Cell 7 first")

print("\n" + "=" * 80)

COMPREHENSIVE PORT NORMALIZATION & DISTANCE DIAGNOSTIC

1. PORT NORMALIZATION CHECK
--------------------------------------------------------------------------------
Cargo: CARGILL_3
Original Load Port: ITAGUAI
Normalized Load Port: ITAGUAI
  → Port name only: ITAGUAI
  → Accent-normalized: ITAGUAI

Original Discharge Port: QINGDAO
Normalized Discharge Port: QINGDAO
  → Port name only: QINGDAO
  → Accent-normalized: QINGDAO

2. DISTANCE LOOKUP TEST FOR CARGILL_3
--------------------------------------------------------------------------------

Laden Leg: ITAGUAI -> QINGDAO
  ✓ Found: 11371.0 nm (exact match)

3. VESSEL ASSIGNMENT CHECK (Cargill + Market)
--------------------------------------------------------------------------------

  Cargill Vessel: ANN BELL
    Current position: QINGDAO
    ETD: 2026-02-25
    ✓ Ballast: 11371.0 nm (exact match)
    ✓ Laden: 11371.0 nm (exact match)
    Arrival: 2026-04-03
    ✓ PROFIT: $1,061,327.64

  Cargill Vessel: OCEAN HORIZON
    Current posit

In [58]:
# Cell 7: Run the optimizer

print("Running optimization...")
print("=" * 60)

assignments_df, total_profit = optimize_assignments(
    cargill_vessels_processed,
    market_vessels_processed,
    cargill_cargoes_processed,
    market_cargoes_processed,
    distance_lookup,
    get_bunker_price,
    get_market_freight_rate
)

print("\n" + "=" * 60)
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Number of Assignments: {len(assignments_df)}")
print("=" * 60)

if len(assignments_df) > 0:
    print("\nAssignments (sorted by profit):")
    print(assignments_df.sort_values('Profit', ascending=False).to_string(index=False))
    
    print("\n\nSummary Statistics:")
    print(f"Average Profit: ${assignments_df['Profit'].mean():,.2f}")
    print(f"Average TCE: ${assignments_df['TCE'].mean():,.2f}")
    print(f"Total Voyage Days: {assignments_df['Voyage_Days'].sum():.1f}")
    
    if assignments_df['Market_Rate'].notna().any():
        print("\nMarket Rate Comparisons:")
        market_comparisons = assignments_df[assignments_df['Market_Rate'].notna()].copy()
        if len(market_comparisons) > 0:
            print(f"  Cargoes with market rate data: {len(market_comparisons)}")
            print(f"  Average opportunity cost: ${market_comparisons['Opportunity_Cost'].mean():,.2f}")
else:
    print("\nNo assignments found. Check constraints and data.")

Running optimization...
Found 25 feasible vessel-cargo pairs
Solution status: OPTIMAL

Total Profit: $6,252,380.55
Number of Assignments: 3

Assignments (sorted by profit):
Vessel_Type      Vessel_Name Cargo_Type  Cargo_ID        Load_Port Discharge_Port       Profit   Revenue        Costs          TCE  Voyage_Days  Days_Ballast  Days_Laden  Offered_Rate  Market_Rate  Opportunity_Cost  Ballast_Distance  Laden_Distance
     Market     IRON CENTURY  Committed CARGILL_1 KAMSAR ANCHORAGE        QINGDAO 3.055681e+06 4140000.0 1.084319e+06 54843.647439    55.716233     16.091233   38.625000          23.0          NaN               NaN           4827.37        11124.00
     Market    CORAL EMPEROR  Committed CARGILL_3          ITAGUAI        QINGDAO 2.632135e+06 4014000.0 1.381865e+06 44386.458607    59.300397     18.236111   39.814286          22.3      21475.0      3861486000.0           5383.30        11370.96
     Market ATLANTIC FORTUNE  Committed CARGILL_2     PORT HEDLAND    LIANYUNGAN

In [59]:
# Optional: Visualize optimal routes on a map

if len(assignments_df) > 0:
    import folium
    
    # Create base map
    m = folium.Map(location=[20, 0], zoom_start=2)
    
    # Color mapping
    colors = {
        'Cargill': 'blue',
        'Market': 'green'
    }
    
    # Add routes
    for _, assignment in assignments_df.iterrows():
        # Get port coordinates
        load_port_name = assignment['Load_Port']
        discharge_port_name = assignment['Discharge_Port']
        
        # Find coordinates from port_locations using raw port names (no normalization)
        load_coords = None
        disc_coords = None
        
        for _, port_row in port_locations.iterrows():
            port_name_raw = str(port_row['port_name']).strip()
            if port_name_raw == str(load_port_name).strip():
                load_coords = [port_row['latitude'], port_row['longitude']]
            if port_name_raw == str(discharge_port_name).strip():
                disc_coords = [port_row['latitude'], port_row['longitude']]
        
        if load_coords and disc_coords:
            # Add line
            folium.PolyLine(
                [load_coords, disc_coords],
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                weight=3,
                opacity=0.7,
                popup=f"{assignment['Vessel_Name']} → {assignment['Cargo_ID']}<br>Profit: ${assignment['Profit']:,.0f}"
            ).add_to(m)
            
            # Add markers
            folium.CircleMarker(
                load_coords,
                radius=8,
                popup=f"Load: {load_port_name}",
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                fill=True
            ).add_to(m)
            
            folium.CircleMarker(
                disc_coords,
                radius=8,
                popup=f"Discharge: {discharge_port_name}",
                color=colors.get(assignment['Vessel_Type'], 'gray'),
                fill=True
            ).add_to(m)
    
    # Save map
    m.save('optimal_routes_map.html')
    print("Map saved to optimal_routes_map.html")
else:
    print("No assignments to visualize")

Map saved to optimal_routes_map.html


In [61]:
# Normalize all port names across all files: City name only, ALL CAPS
import unicodedata

def normalize_port_name_to_city_only(port_str):
    """
    Extract city name only (before underscore), remove accents, 
    convert to uppercase, keep spaces (as in Port Distances format)
    """
    if pd.isna(port_str):
        return None
    
    text = str(port_str).strip().strip('"')
    
    # Remove accents
    nfd = unicodedata.normalize('NFD', text)
    no_accents = ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')
    
    # Extract city name only (before underscore)
    if '_' in no_accents:
        city_name = no_accents.split('_')[0].strip()
    else:
        city_name = no_accents.strip()
    
    # Convert to uppercase
    normalized = city_name.upper()
    
    return normalized

print("Normalizing port names across all files...")
print("=" * 60)

# 1. Update port_locations.csv
print("\n1. Updating port_locations.csv...")
port_locations = pd.read_csv(base / "port_locations.csv")
port_locations['port_name'] = port_locations['port_name'].apply(normalize_port_name_to_city_only)
port_locations.to_csv(base / "port_locations.csv", index=False)
print(f"   ✓ Updated {len(port_locations)} ports")
print(f"   Sample: {port_locations['port_name'].head(5).tolist()}")

# 2. Update Cargill_Capesize_Vessels.csv
print("\n2. Updating Cargill_Capesize_Vessels.csv...")
cargill_vessels = pd.read_csv(base / "Cargill_Capesize_Vessels.csv")
cargill_vessels['Current Position / Status'] = cargill_vessels['Current Position / Status'].apply(normalize_port_name_to_city_only)
cargill_vessels.to_csv(base / "Cargill_Capesize_Vessels.csv", index=False)
print(f"   ✓ Updated {len(cargill_vessels)} vessels")
print(f"   Sample: {cargill_vessels['Current Position / Status'].tolist()}")

# 3. Update Market_Vessels_Formatted.csv
print("\n3. Updating Market_Vessels_Formatted.csv...")
market_vessels = pd.read_csv(base / "Market_Vessels_Formatted.csv")
market_vessels['Current Position / Status'] = market_vessels['Current Position / Status'].apply(normalize_port_name_to_city_only)
market_vessels.to_csv(base / "Market_Vessels_Formatted.csv", index=False)
print(f"   ✓ Updated {len(market_vessels)} vessels")
print(f"   Sample: {market_vessels['Current Position / Status'].head(5).tolist()}")

# 4. Update Cargill_Committed_Cargoes_Structured.csv
print("\n4. Updating Cargill_Committed_Cargoes_Structured.csv...")
cargill_cargoes = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")
cargill_cargoes['Load_Port_Primary'] = cargill_cargoes['Load_Port_Primary'].apply(normalize_port_name_to_city_only)
cargill_cargoes['Discharge_Port_Primary'] = cargill_cargoes['Discharge_Port_Primary'].apply(normalize_port_name_to_city_only)
cargill_cargoes.to_csv(base / "Cargill_Committed_Cargoes_Structured.csv", index=False)
print(f"   ✓ Updated {len(cargill_cargoes)} cargoes")
print(f"   Load ports: {cargill_cargoes['Load_Port_Primary'].tolist()}")
print(f"   Discharge ports: {cargill_cargoes['Discharge_Port_Primary'].tolist()}")

# 5. Update Market_Cargoes_Structured.csv
print("\n5. Updating Market_Cargoes_Structured.csv...")
market_cargoes = pd.read_csv(base / "Market_Cargoes_Structured.csv")
market_cargoes['Load_Port'] = market_cargoes['Load_Port'].apply(normalize_port_name_to_city_only)
market_cargoes['Discharge_Port'] = market_cargoes['Discharge_Port'].apply(normalize_port_name_to_city_only)
market_cargoes.to_csv(base / "Market_Cargoes_Structured.csv", index=False)
print(f"   ✓ Updated {len(market_cargoes)} cargoes")
print(f"   Load ports: {market_cargoes['Load_Port'].head(5).tolist()}")
print(f"   Discharge ports: {market_cargoes['Discharge_Port'].head(5).tolist()}")

print("\n" + "=" * 60)
print("✓ All port names normalized successfully!")
print("All ports are now in format: CITY NAME ONLY, ALL CAPS")
print("=" * 60)

Normalizing port names across all files...

1. Updating port_locations.csv...
   ✓ Updated 28 ports
   Sample: ['DAMPIER', 'PONTA DA MADEIRA', 'SALDANHA BAY', 'TABONEO', 'VANCOUVER (CANADA)']

2. Updating Cargill_Capesize_Vessels.csv...
   ✓ Updated 4 vessels
   Sample: ['QINGDAO', 'MAP TA PHUT', 'GWANGYANG LNG TERMINAL', 'FANGCHENG']

3. Updating Market_Vessels_Formatted.csv...
   ✓ Updated 11 vessels
   Sample: ['PARADIP', 'CAOFEIDIAN', 'ROTTERDAM', 'XIAMEN', 'KANDLA']

4. Updating Cargill_Committed_Cargoes_Structured.csv...
   ✓ Updated 3 cargoes
   Load ports: ['KAMSAR ANCHORAGE', 'PORT HEDLAND', 'ITAGUAI']
   Discharge ports: ['QINGDAO', 'LIANYUNGANG', 'QINGDAO']

5. Updating Market_Cargoes_Structured.csv...
   ✓ Updated 8 cargoes
   Load ports: ['DAMPIER', 'PONTA DA MADEIRA', 'SALDANHA BAY', 'TABONEO', 'VANCOUVER (CANADA)']
   Discharge ports: ['QINGDAO', 'CAOFEIDIAN', 'TIANJIN', 'KRISHNAPATNAM', 'FANGCHENG']

✓ All port names normalized successfully!
All ports are now in format:

In [62]:
# Diagnostic: Verify ALL cargo routes have a distance in Port Distances (with runtime logs)
import json, time
from pathlib import Path

LOG_PATH = Path(".cursor/debug.log")

def _log(hypothesisId, location, message, data=None, runId="route-distance-check"):
    payload = {
        "id": f"log_{int(time.time()*1000)}",
        "timestamp": int(time.time()*1000),
        "sessionId": "debug-session",
        "runId": runId,
        "hypothesisId": hypothesisId,
        "location": location,
        "message": message,
        "data": data or {},
    }
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    with LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(payload) + "\n")

# Hypotheses:
# H1: Port strings still not normalized to Port Distances format (spaces/all caps)
# H2: Distance exists but only in reverse direction
# H3: Distance truly missing from Port Distances
# H4: Hidden whitespace/punctuation causes mismatches

_log("H1", "diag:routes:1", "Starting route distance verification")

# Helper: normalize to Port Distances style (ALL CAPS, spaces, stripped)
def norm_pd(name):
    if pd.isna(name):
        return None
    return str(name).strip().strip('"').upper()

# Ensure we have a distance lookup that uses Port Distances format
# (If you already ran Cell 2 after normalization, distance_lookup should exist)
if 'distance_lookup' not in globals() or not isinstance(distance_lookup, dict) or len(distance_lookup) == 0:
    _log("H1", "diag:routes:2", "distance_lookup missing/empty; rebuilding from Port Distances.csv")
    pdist = pd.read_csv(base / "Port Distances.csv")
    distance_lookup = {}
    for _, r in pdist.iterrows():
        a = norm_pd(r["PORT_NAME_FROM"])
        b = norm_pd(r["PORT_NAME_TO"])
        d = float(r["DISTANCE"]) if pd.notna(r["DISTANCE"]) else None
        if a and b and d is not None:
            distance_lookup[(a, b)] = d
            distance_lookup[(b, a)] = d

_log("H1", "diag:routes:3", "distance_lookup ready", {"entries": len(distance_lookup)})

# Collect routes from the 2 cargo files (raw + processed if available)
route_records = []

# Cargill committed
cargill_df = pd.read_csv(base / "Cargill_Committed_Cargoes_Structured.csv")
for i, r in cargill_df.iterrows():
    route_records.append({
        "source": "cargill_committed",
        "row": int(i),
        "route": r.get("Route"),
        "load": r.get("Load_Port_Primary"),
        "disc": r.get("Discharge_Port_Primary"),
    })

# Market cargoes
market_df = pd.read_csv(base / "Market_Cargoes_Structured.csv")
for i, r in market_df.iterrows():
    route_records.append({
        "source": "market",
        "row": int(i),
        "route": r.get("Route"),
        "load": r.get("Load_Port"),
        "disc": r.get("Discharge_Port"),
    })

_log("H1", "diag:routes:4", "Loaded cargo routes", {"count": len(route_records)})

# Check routes
results = []
missing = []

for rec in route_records:
    a_raw = rec["load"]
    b_raw = rec["disc"]
    a = norm_pd(a_raw)
    b = norm_pd(b_raw)

    if not a or not b:
        results.append({**rec, "load_norm": a, "disc_norm": b, "status": "missing_port"})
        continue

    d = distance_lookup.get((a, b))
    if d is not None:
        results.append({**rec, "load_norm": a, "disc_norm": b, "status": "found", "distance_nm": d})
    else:
        # We store both directions in the dict, so if it's missing, it’s truly missing (H3)
        results.append({**rec, "load_norm": a, "disc_norm": b, "status": "missing_distance"})
        missing.append((a, b))

res_df = pd.DataFrame(results)

found_n = int((res_df["status"] == "found").sum())
miss_n = int((res_df["status"] == "missing_distance").sum())
miss_port_n = int((res_df["status"] == "missing_port").sum())

_log("H3", "diag:routes:5", "Route distance check complete", {
    "found": found_n,
    "missing_distance": miss_n,
    "missing_port": miss_port_n,
})

print("="*80)
print("ROUTE DISTANCE AVAILABILITY SUMMARY")
print("="*80)
print(f"Total routes checked: {len(res_df)}")
print(f"Found distances:      {found_n}")
print(f"Missing distances:    {miss_n}")
print(f"Missing ports:        {miss_port_n}")

if miss_n:
    print("\nMissing distance routes (showing up to 50):")
    show = res_df[res_df["status"] == "missing_distance"][
        ["source", "row", "route", "load_norm", "disc_norm"]
    ].head(50)
    print(show.to_string(index=False))
    _log("H3", "diag:routes:6", "Sample missing distance routes", {
        "sample": show.to_dict(orient="records")
    })

if miss_port_n:
    print("\nMissing port fields (showing up to 50):")
    showp = res_df[res_df["status"] == "missing_port"][
        ["source", "row", "route", "load", "disc"]
    ].head(50)
    print(showp.to_string(index=False))
    _log("H1", "diag:routes:7", "Sample missing port fields", {
        "sample": showp.to_dict(orient="records")
    })

print("\nTip: If many are missing, confirm ports are CITY-ONLY ALL-CAPS and that you re-ran Cell 2.")
print("Log written to .cursor/debug.log")

ROUTE DISTANCE AVAILABILITY SUMMARY
Total routes checked: 11
Found distances:      11
Missing distances:    0
Missing ports:        0

Tip: If many are missing, confirm ports are CITY-ONLY ALL-CAPS and that you re-ran Cell 2.
Log written to .cursor/debug.log


In [63]:
# Check: every port in port_locations appears in Port Distances at least once (FROM or TO)
import pandas as pd

from pathlib import Path
base = Path("data")

ports = pd.read_csv(base / "port_locations.csv")
dist = pd.read_csv(base / "Port Distances.csv")

# Normalize to match Port Distances style
def norm(x):
    if pd.isna(x):
        return None
    return str(x).strip().strip('"').upper()

ports_set = set(norm(p) for p in ports["port_name"] if pd.notna(p))

from_set = set(norm(x) for x in dist["PORT_NAME_FROM"] if pd.notna(x))
to_set   = set(norm(x) for x in dist["PORT_NAME_TO"] if pd.notna(x))

dist_ports = from_set | to_set

missing = sorted(p for p in ports_set if p not in dist_ports)

print("=" * 80)
print("PORT LOCATIONS vs PORT DISTANCES")
print("=" * 80)
print(f"Ports in port_locations.csv: {len(ports_set)}")
print(f"Unique ports in Port Distances.csv (FROM∪TO): {len(dist_ports)}")
print(f"Ports missing from Port Distances: {len(missing)}")

if missing:
    print("\nMissing ports (showing up to 200):")
    for p in missing[:200]:
        print(" -", p)
else:
    print("\n✓ All ports from port_locations.csv appear at least once in Port Distances.csv")

PORT LOCATIONS vs PORT DISTANCES
Ports in port_locations.csv: 28
Unique ports in Port Distances.csv (FROM∪TO): 1950
Ports missing from Port Distances: 0

✓ All ports from port_locations.csv appear at least once in Port Distances.csv


In [64]:
# Diagnostic: Find ALL missing port-to-port distances needed for optimization
print("=" * 80)
print("MISSING PORT-TO-PORT DISTANCES REPORT")
print("=" * 80)

# Build distance lookup from Port Distances.csv (using raw port names)
distance_lookup = {}
for _, row in port_distances.iterrows():
    port_from = str(row['PORT_NAME_FROM']).strip()
    port_to = str(row['PORT_NAME_TO']).strip()
    distance = row['DISTANCE']
    
    key = (port_from, port_to)
    distance_lookup[key] = distance
    
    # Also store reverse direction
    key_reverse = (port_to, port_from)
    if key_reverse not in distance_lookup:
        distance_lookup[key_reverse] = distance

# Collect all vessel positions
vessel_positions = {}
for _, row in cargill_vessels.iterrows():
    pos = str(row['Current Position / Status']).strip()
    vessel_name = row['Vessel Name']
    if pos:
        vessel_positions[vessel_name] = pos

for _, row in market_vessels.iterrows():
    pos = str(row['Current Position / Status']).strip()
    vessel_name = row['Vessel Name']
    if pos:
        vessel_positions[vessel_name] = pos

# Collect all cargo load/discharge ports
cargo_routes = []
for _, row in cargill_cargoes.iterrows():
    load = str(row['Load_Port_Primary']).strip()
    disc = str(row['Discharge_Port_Primary']).strip()
    route = row['Route']
    cargo_id = f"CARGILL_{row.name + 1}"
    if load and disc:
        cargo_routes.append({
            'cargo_id': cargo_id,
            'route': route,
            'load_port': load,
            'discharge_port': disc
        })

for _, row in market_cargoes.iterrows():
    load = str(row['Load_Port']).strip()
    disc = str(row['Discharge_Port']).strip()
    route = row['Route']
    cargo_id = f"MARKET_{row.name + 1}"
    if load and disc:
        cargo_routes.append({
            'cargo_id': cargo_id,
            'route': route,
            'load_port': load,
            'discharge_port': disc
        })

# Check all vessel-cargo combinations
missing_distances = []

for vessel_name, vessel_pos in vessel_positions.items():
    for cargo in cargo_routes:
        load_port = cargo['load_port']
        disc_port = cargo['discharge_port']
        
        # Check ballast leg: vessel position -> load port
        ballast_key = (vessel_pos, load_port)
        ballast_reverse = (load_port, vessel_pos)
        ballast_distance = distance_lookup.get(ballast_key) or distance_lookup.get(ballast_reverse)
        
        if ballast_distance is None:
            missing_distances.append({
                'type': 'BALLAST',
                'vessel': vessel_name,
                'vessel_position': vessel_pos,
                'cargo_id': cargo['cargo_id'],
                'route': cargo['route'],
                'from_port': vessel_pos,
                'to_port': load_port,
                'leg': f"{vessel_pos} → {load_port}"
            })
        
        # Check laden leg: load port -> discharge port
        laden_key = (load_port, disc_port)
        laden_reverse = (disc_port, load_port)
        laden_distance = distance_lookup.get(laden_key) or distance_lookup.get(laden_reverse)
        
        if laden_distance is None:
            missing_distances.append({
                'type': 'LADEN',
                'vessel': vessel_name,
                'vessel_position': vessel_pos,
                'cargo_id': cargo['cargo_id'],
                'route': cargo['route'],
                'from_port': load_port,
                'to_port': disc_port,
                'leg': f"{load_port} → {disc_port}"
            })

# Group and summarize
if missing_distances:
    missing_df = pd.DataFrame(missing_distances)
    
    print(f"\nTotal missing distances found: {len(missing_distances)}")
    print(f"  - Ballast legs missing: {len(missing_df[missing_df['type'] == 'BALLAST'])}")
    print(f"  - Laden legs missing: {len(missing_df[missing_df['type'] == 'LADEN'])}")
    
    # Group by leg (unique from→to pairs)
    unique_missing_legs = {}
    for _, row in missing_df.iterrows():
        leg_key = (row['from_port'], row['to_port'])
        if leg_key not in unique_missing_legs:
            unique_missing_legs[leg_key] = {
                'from': row['from_port'],
                'to': row['to_port'],
                'leg': row['leg'],
                'type': row['type'],
                'affected_vessels': [],
                'affected_cargoes': []
            }
        unique_missing_legs[leg_key]['affected_vessels'].append(row['vessel'])
        unique_missing_legs[leg_key]['affected_cargoes'].append(row['cargo_id'])
    
    print(f"\nUnique missing port-to-port distances: {len(unique_missing_legs)}")
    print("\n" + "=" * 80)
    print("DETAILED LIST OF MISSING DISTANCES")
    print("=" * 80)
    
    for i, (leg_key, info) in enumerate(sorted(unique_missing_legs.items()), 1):
        print(f"\n{i}. {info['leg']} ({info['type']} leg)")
        print(f"   Affects {len(set(info['affected_vessels']))} vessel(s): {', '.join(set(info['affected_vessels']))}")
        print(f"   Affects {len(set(info['affected_cargoes']))} cargo(es): {', '.join(set(info['affected_cargoes']))}")
    
    # Export to CSV for easy reference
    export_df = pd.DataFrame([
        {
            'PORT_NAME_FROM': info['from'],
            'PORT_NAME_TO': info['to'],
            'LEG_TYPE': info['type'],
            'AFFECTED_VESSELS': ', '.join(set(info['affected_vessels'])),
            'AFFECTED_CARGOES': ', '.join(set(info['affected_cargoes']))
        }
        for info in unique_missing_legs.values()
    ])
    
    export_df.to_csv('missing_port_distances.csv', index=False)
    print(f"\n✓ Exported detailed list to: missing_port_distances.csv")
    
else:
    print("\n✓ All required port-to-port distances are available in Port Distances.csv!")

print("\n" + "=" * 80)

MISSING PORT-TO-PORT DISTANCES REPORT

✓ All required port-to-port distances are available in Port Distances.csv!

